Voice Frequency

In [1]:
import librosa
import numpy as np

In [2]:
def analyze_voice_pitch(audio_path):
    y, sr = librosa.load(audio_path, sr=None)

    # Estimate pitch using pyin
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),   # low human voice range
        fmax=librosa.note_to_hz("C7")    # high human voice range
    )

    # Remove empty/unvoiced frames
    valid_pitch = f0[~np.isnan(f0)]

    if len(valid_pitch) == 0:
        return {
            "avg_pitch": 0,
            "min_pitch": 0,
            "max_pitch": 0,
            "pitch_range": 0,
            "pitch_variation": 0
        }

    avg_pitch = float(np.mean(valid_pitch))
    min_pitch = float(np.min(valid_pitch))
    max_pitch = float(np.max(valid_pitch))
    pitch_range = max_pitch - min_pitch
    pitch_variation = float(np.std(valid_pitch))

    return {
        "avg_pitch": round(avg_pitch, 2),
        "min_pitch": round(min_pitch, 2),
        "max_pitch": round(max_pitch, 2),
        "pitch_range": round(pitch_range, 2),
        "pitch_variation": round(pitch_variation, 2)
    }

- avg_pitch        = average vocal pitch
- min_pitch        = lowest voice pitch
- max_pitch        = highest voice pitch
- pitch_range      = emotional expressiveness / variation
- pitch_variation  = instability or expressiveness

In [3]:
voice_data = analyze_voice_pitch("test_voice.wav")
print(voice_data)

c:\Users\Yun Nee\miniconda3\envs\fyppro\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'avg_pitch': 279.12, 'min_pitch': 217.47, 'max_pitch': 385.26, 'pitch_range': 167.79, 'pitch_variation': 35.59}


# RealTime pitch and energy

In [4]:
import sounddevice as sd
import numpy as np
import librosa
# from pythonosc.udp_client import SimpleUDPClient

# client = SimpleUDPClient("127.0.0.1", 8000)

sample_rate = 16000
block_duration = 0.5  # seconds
block_size = int(sample_rate * block_duration)

def audio_callback(indata, frames, time, status):
    audio = indata[:, 0].astype(np.float32)

    # --------------------
    # ENERGY
    # --------------------
    rms = np.sqrt(np.mean(audio ** 2))
    energy = min(rms * 20, 1.0)

    # --------------------
    # PITCH
    # --------------------
    try:
        f0 = librosa.yin(
            audio,
            fmin=80,
            fmax=400,
            sr=sample_rate
        )

        pitch = float(np.median(f0))

        if energy < 0.02:
            pitch = 0

    except:
        pitch = 0

    pitch_norm = np.clip((pitch - 80) / (400 - 80), 0, 1) if pitch > 0 else 0

    # client.send_message("/voice/energy", float(energy))
    # client.send_message("/voice/pitch", float(pitch))
    # client.send_message("/voice/pitch_norm", float(pitch_norm))

    print("energy:", round(energy, 3), "pitch:", round(pitch, 1))

with sd.InputStream(
    channels=1,
    samplerate=sample_rate,
    blocksize=block_size,
    callback=audio_callback
):
    print("Listening... Press Ctrl+C to stop.")
    while True:
        pass

Listening... Press Ctrl+C to stop.
energy: 0.0 pitch: 0
energy: 0.021 pitch: 175.0
energy: 0.011 pitch: 0
energy: 0.008 pitch: 0
energy: 0.006 pitch: 0
energy: 0.12 pitch: 203.6
energy: 1.0 pitch: 186.0
energy: 0.325 pitch: 193.6
energy: 0.003 pitch: 0
energy: 0.592 pitch: 184.7
energy: 0.256 pitch: 182.8
energy: 0.012 pitch: 0
energy: 0.552 pitch: 189.4
energy: 0.069 pitch: 154.5
energy: 0.003 pitch: 0
energy: 0.016 pitch: 0
energy: 1.0 pitch: 278.1
energy: 0.05 pitch: 159.3
energy: 0.31 pitch: 147.7
energy: 0.648 pitch: 317.3
energy: 0.606 pitch: 318.4
energy: 0.793 pitch: 312.5
energy: 0.843 pitch: 307.1
energy: 1.0 pitch: 400.0
energy: 0.631 pitch: 285.7
energy: 0.492 pitch: 283.0
energy: 0.478 pitch: 316.6
energy: 0.467 pitch: 313.2
energy: 0.071 pitch: 247.0
energy: 0.003 pitch: 0
energy: 0.003 pitch: 0
energy: 0.003 pitch: 0
energy: 0.004 pitch: 0
energy: 0.003 pitch: 0
energy: 0.759 pitch: 200.4
energy: 0.004 pitch: 0
energy: 0.003 pitch: 0
energy: 0.003 pitch: 0
energy: 0.008 

KeyboardInterrupt: 